<a href="https://colab.research.google.com/github/efecclick/qwen3/blob/main/qwen3_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install transformers
!pip install accelerate
!pip install qwen-vl-utils
!pip install pillow
!pip install evaluate

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
import torch

device = "cpu" # Cihazı CPU olarak ayarla

# Qwen3-VL-4B-Instruct modeli
model_instruct_id = "Qwen/Qwen3-VL-4B-Instruct"

# Qwen3-VL-8B-Instruct modeli (Karşılaştırmak için)
model_thinking_id = "Qwen/Qwen3-VL-8B-Instruct"

# Model 1: Qwen3-VL-4B-Instruct
# CPU üzerinde 4-bit niceleme uygun değildir, bu yüzden kaldırıldı.
model1 = Qwen3VLForConditionalGeneration.from_pretrained(
    model_instruct_id,
    torch_dtype=torch.float32, # CPU için float32 kullan
)

process1 = AutoProcessor.from_pretrained(
    model_instruct_id,
)

# Model 3: Qwen3-VL-8B-Instruct
# CPU üzerinde 4-bit niceleme uygun değildir, bu yüzden kaldırıldı.
model3 = Qwen3VLForConditionalGeneration.from_pretrained(
    model_thinking_id,
    torch_dtype=torch.float32, # CPU için float32 kullan
)

process3 = AutoProcessor.from_pretrained(
    model_thinking_id,
)

# YOLO modeli GPU'ya bağımlı olduğu için kaldırıldı.

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

In [2]:
import os
import re
import json
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Define a mapping for synonyms to help with ground truth matching
CATEGORY_SYNONYMS = {
    'potted plant': ['cactus', 'pot', 'plant'],
    'telephone': ['telephone', 'phone', 'red phone'],
    'lamp': ['lamp', 'red lamp'],
    'pushpin': ['pushpin', 'pin'],
    'donut': ['donut', 'doughnut'],
    'pencil': ['pencil', 'pen'],
    'eraser': ['eraser'],
    'glasses': ['glasses'],
    'clock': ['clock'],
    'dining table': ['dining table', 'table', 'desk', 'plate'],
}

def parse_qwen_boxes(output_text, img_w, img_h):
    """
    Qwen modelinin ürettiği <|box_start|>(ymin,xmin,ymax,xmax)<|box_end|>**nesne**
    formatındaki koordinatları yakalar ve piksel koordinatlarına dönüştürür.
    Ayrıca, `Qwen3-VL-8B-Instruct` modelinin JSON çıktı formatını da işler.
    """
    predictions = []

    # Pattern for Qwen3-VL-4B-Instruct style output (requested format)
    # <|box_start|>(ymin,xmin,ymax,xmax)<|box_end|>**object_name**
    pattern_4b = r"<\|box_start\|>\s*\((\d+),(\d+),(\d+),(\d+)\)\s*<\|box_end\|>(?:\*\*([^\*]+?)\*\*)?([^<\n]*)"
    matches_4b = re.findall(pattern_4b, output_text)

    for match in matches_4b:
        # Correctly parse Qwen's (ymin,xmin,ymax,xmax) format
        xmin_qwen, ymin_qwen, xmax_qwen, ymax_qwen = map(int, match[0:4])

        cat_name = match[4] if match[4] else match[5]
        cat_name = cat_name.strip().lower()

        # Qwen koordinatları 1000 üzerinden normalize ettiği için gerçek piksele çeviriyoruz
        abs_xmin = int((xmin_qwen / 1000.0) * img_w)
        abs_ymin = int((ymin_qwen / 1000.0) * img_h)
        abs_xmax = int((xmax_qwen / 1000.0) * img_w)
        abs_ymax = int((ymax_qwen / 1000.0) * img_h)

        predictions.append({
            'category': cat_name,
            'box': [abs_xmin, abs_ymin, abs_xmax, abs_ymax]
        })

    # Pattern for Qwen3-VL-8B-Instruct's JSON output format
    # It's wrapped in ```json ... ```
    json_match = re.search(r"```json\n([\s\S]*?)\n```", output_text)
    if json_match:
        json_str = json_match.group(1)
        try:
            json_data = json.loads(json_str)
            for item in json_data:
                if 'bbox_2d' in item and 'label' in item:
                    ymin_qwen, xmin_qwen, ymax_qwen, xmax_qwen = map(int, item['bbox_2d'])
                    cat_name = item['label'].strip().lower()

                    # Coordinates are also normalized to 1000
                    abs_xmin = int((xmin_qwen / 1000.0) * img_w)
                    abs_ymin = int((ymin_qwen / 1000.0) * img_h)
                    abs_xmax = int((xmax_qwen / 1000.0) * img_w)
                    abs_ymax = int((ymax_qwen / 1000.0) * img_h)

                    new_prediction = {
                        'category': cat_name,
                        'box': [abs_xmin, abs_ymin, abs_xmax, abs_ymax]
                    }
                    # Add only if not already present (to avoid duplicates from other patterns if they somehow matched)
                    if new_prediction not in predictions:
                        predictions.append(new_prediction)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error for 8B model output: {e}")

    return predictions

def plot_dual_boxes(image_path, qwen_preds, qwen8b_preds):
    """
    Sol tarafa Qwen3-VL-4B-Instruct, sağ tarafa Qwen3-VL-8B-Instruct tahminlerini kutularıyla çizer.
    """
    img = Image.open(image_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # Sol Grafik - Qwen3-VL-4B-Instruct
    ax1.imshow(img)
    ax1.set_title("Model 1: Qwen3-VL 4B Instruct Tahminleri (Kırmızı)", fontsize=14, color='red')
    for p in qwen_preds:
        box = p['box']
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                 linewidth=2, edgecolor='r', facecolor='none')
        ax1.add_patch(rect)
        ax1.text(box[0], box[1] - 5, p['category'], color='red', fontsize=10,
                 weight='bold', bbox=dict(facecolor='white', alpha=0.6, pad=2))

    # Sağ Grafik - Qwen3-VL-8B-Instruct
    ax2.imshow(img)
    ax2.set_title("Model 3: Qwen3-VL 8B Instruct Tahminleri (Mavi)", fontsize=14, color='blue')
    for p in qwen8b_preds:
        box = p['box']
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                 linewidth=2, edgecolor='b', facecolor='none')
        ax2.add_patch(rect)
        ax2.text(box[0], box[1] - 5, p['category'], color='blue', fontsize=10,
                 weight='bold', bbox=dict(facecolor='white', alpha=0.6, pad=2))

    plt.tight_layout()
    plt.show()

def hesapla_model_metrikleri(model_predictions, ground_truth):
    """
    Tahminleri ground truth ile kıyaslayıp 3 temel metriği döner.
    model_predictions: list of {'category': 'cat_name', 'box': [...]}
    ground_truth: dict of {'gt_cat_name': count}
    """
    basarili_eslesmeler = Counter() # Counts how many predictions successfully matched a GT category

    gt_categories_in_scope = list(ground_truth.keys())

    # Create a mutable copy of ground_truth counts to consume as predictions are matched
    temp_gt_available = ground_truth.copy()

    false_positive_categories = Counter() # For predictions that don't match any remaining GT

    for pred in model_predictions:
        pred_cat_lower = pred['category'].lower()
        matched_gt_cat = None

        # Try to find a ground truth match for the predicted category
        for gt_cat in gt_categories_in_scope:
            # Check for direct match or synonym match
            synonyms_for_gt = CATEGORY_SYNONYMS.get(gt_cat, [])
            if pred_cat_lower == gt_cat.lower() or pred_cat_lower in [s.lower() for s in synonyms_for_gt]:
                if temp_gt_available.get(gt_cat, 0) > 0: # Only match if there's an available GT instance
                    matched_gt_cat = gt_cat
                    break # Found a match, consume it and move to next prediction

        if matched_gt_cat:
            basarili_eslesmeler[matched_gt_cat] += 1
            temp_gt_available[matched_gt_cat] -= 1 # Consume one instance from available GT
        else:
            # If no match found, it's a false positive
            if pred_cat_lower: # Only count as FP if category name is not empty
                false_positive_categories[pred_cat_lower] += 1

    basarili_metric = {}
    kacirilan_metric = {}
    false_positive_metric = {}

    # Populate basarili and kacirilan metrics
    for gt_cat, original_gt_count in ground_truth.items():
        matched_count = basarili_eslesmeler.get(gt_cat, 0)

        if matched_count > 0:
            basarili_metric[gt_cat] = f"({min(matched_count, original_gt_count)}/{original_gt_count})"
        else:
            kacirilan_metric[gt_cat] = f"(0/{original_gt_count})"

        # If the model found more items than the ground truth for a category, the excess are false positives
        if matched_count > original_gt_count:
            false_positive_metric[f"Excess {gt_cat} (over-detection)"] = false_positive_metric.get(f"Excess {gt_cat} (over-detection)", 0) + (matched_count - original_gt_count)

    # Add other false positives (predictions that didn't match any GT category or were not consumed)
    for fp_cat, fp_count in false_positive_categories.items():
        false_positive_metric[fp_cat] = false_positive_metric.get(fp_cat, 0) + fp_count

    return basarili_metric, kacirilan_metric, false_positive_metric

def draw_and_analyze_dual(image_path, qwen_preds, qwen8b_preds, ground_truth):
    """
    Raporu tamamen kullanıcının belirlediği ground_truth listesi üzerinden oluşturur.
    """
    gt_categories_in_scope = list(ground_truth.keys())

    # Prepare counts for the table, using the mapping logic
    qwen_table_counts = Counter() # Tracks successful matches for table display
    temp_gt_qwen = ground_truth.copy() # Temporary GT for Qwen table matching

    for p in qwen_preds:
        pred_cat_lower = p['category'].lower()
        matched_gt = None
        for gt_cat in gt_categories_in_scope:
            synonyms_for_gt = CATEGORY_SYNONYMS.get(gt_cat, [])
            if (pred_cat_lower == gt_cat.lower() or pred_cat_lower in [s.lower() for s in synonyms_for_gt]) and temp_gt_qwen.get(gt_cat, 0) > 0:
                matched_gt = gt_cat
                break
        if matched_gt:
            qwen_table_counts[matched_gt] += 1
            temp_gt_qwen[matched_gt] -= 1 # Consume one GT instance for table display

    qwen8b_table_counts = Counter() # Tracks successful matches for table display
    temp_gt_qwen8b = ground_truth.copy() # Temporary GT for Qwen 8B table matching

    for p in qwen8b_preds:
        pred_cat_lower = p['category'].lower()
        matched_gt = None
        for gt_cat in gt_categories_in_scope:
            synonyms_for_gt = CATEGORY_SYNONYMS.get(gt_cat, [])
            if (pred_cat_lower == gt_cat.lower() or pred_cat_lower in [s.lower() for s in synonyms_for_gt]) and temp_gt_qwen8b.get(gt_cat, 0) > 0:
                matched_gt = gt_cat
                break
        if matched_gt:
            qwen8b_table_counts[matched_gt] += 1
            temp_gt_qwen8b[matched_gt] -= 1 # Consume one GT instance for table display

    print("="*75)
    print("      ÇİFT MODELLİ BENCHMARK PERFORMANS VE BAŞARI ORANI RAPORU")
    print("="*75)
    print(f"{'NESNE ADI':<25} | {'QWEN3-VL 4B Instruct BAŞARISI':<22} | {'QWEN3-VL 8B Instruct BAŞARISI':<22}")
    print("-"*75)

    for nesne in ground_truth.keys():
        gt_adet = ground_truth[nesne]

        qwen_adet = qwen_table_counts.get(nesne, 0)
        qwen8b_adet = qwen8b_table_counts.get(nesne, 0)

        # Display how many were found versus how many should have been found (up to GT count)
        qwen_str = f"({min(qwen_adet, gt_adet)}/{gt_adet}) Buldu" if qwen_adet > 0 else "(0) Iskaladı"
        qwen8b_str = f"({min(qwen8b_adet, gt_adet)}/{gt_adet}) Buldu" if qwen8b_adet > 0 else "(0) Iskaladı"

        print(f"{nesne:<25} | {qwen_str:<22} | {qwen8b_str:<22}")

    print("="*75)

    # Calculate detailed metrics using the refined function
    qwen_b, qwen_k, qwen_fp = hesapla_model_metrikleri(qwen_preds, ground_truth)
    qwen8b_b, qwen8b_k, qwen8b_fp = hesapla_model_metrikleri(qwen8b_preds, ground_truth)

    print("\n[QWEN3-VL 4B Instruct] DETAYLI METRİKLER:")
    print(f"  * Başarıyla Eşleşen Nesneler (Oranlar) : {qwen_b}")
    print(f"  * Modelin Kaçırdığı Nesneler          : {qwen_k}")
    print(f"  * False Positive (Ekstra Bulunanlar)  : {qwen_fp}")

    print("\n[QWEN3-VL 8B Instruct] DETAYLI METRİKLER:")
    print(f"  * Başarıyla Eşleşen Nesneler (Oranlar) : {qwen8b_b}")
    print(f"  * Modelin Kaçırdığı Nesneler          : {qwen8b_k}")
    print(f"  * False Positive (Ekstra Bulunanlar)  : {qwen8b_fp}")
    print("="*75)

In [1]:
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
import time # Zaman ölçümü için eklendi

# --- AYARLAR VE MANUEL GROUND TRUTH ---
TEST_IMAGE_PATH = "/content/images.jpg"  # Colab'deki resim yolu

manuel_ground_truth = {
    "potted plant": 1,
    "telephone": 1,
    "clock": 1,
    "lamp": 1,
    "glasses": 1,
    "pencil": 1,
    "eraser": 1,
    "pushpin": 1,
    "dining table": 1
}
# ---------------------------------------

# Ortak Prompt (4B modeli için kullanılacak)
DETECTION_PROMPT_4B = (
    "Detect the objects in the image. For each object, return its bounding box coordinates "
    "and name in the exact format: <|box_start|>(ymin,xmin,ymax,xmax)<|box_end|>**object_name**."
)

# 8B modeli için daha zorlayıcı bir Prompt (JSON formatında çıktı isteme)
DETECTION_PROMPT_8B = (
    "Detect ALL objects in the image and provide their bounding box coordinates and names. "
    "For EVERY object you detect, you MUST return its bounding box coordinates and name as a JSON list of dictionaries. "
    "Each dictionary should have 'bbox_2d' (as [ymin, xmin, ymax, xmax] from 0 to 1000) and 'label' (as object name) keys. "
    "Example: ```json\n[\n\t{\"bbox_2d\": [0, 0, 1000, 999], \"label\": \"desk\"}\n]\n```"
)


img_pil = Image.open(TEST_IMAGE_PATH)
img_w, img_h = img_pil.size

# CPU ortamında nvidia-smi çalışmadığı için kaldırıldı
# print("\n--- GPU Kullanımı (Inference Öncesi) ---")
# !nvidia-smi

# Model 1: QWEN3-VL-Instruct TAHMİN ADIMI (4B model)
messages_instruct = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": TEST_IMAGE_PATH},
            {"type": "text", "text": DETECTION_PROMPT_4B}
        ]
    }
]

text_instruct = process1.apply_chat_template(messages_instruct, tokenize=False, add_generation_prompt=True)
image_inputs_instruct, video_inputs_instruct = process_vision_info(messages_instruct)
inputs_instruct = process1(text=[text_instruct], images=image_inputs_instruct, videos=video_inputs_instruct, padding=True, return_tensors="pt")

if "pixel_values" in inputs_instruct:
    inputs_instruct["pixel_values"] = inputs_instruct["pixel_values"].to(torch.float32) # CPU için float32 kullan

inputs_instruct = inputs_instruct.to(device)

start_time_4b = time.perf_counter() # 4B model çıkarım başlangıç zamanı
with torch.no_grad():
    # torch.amp.autocast CPU'da desteklenmediği için kaldırıldı
    generated_ids_instruct = model1.generate(**inputs_instruct, max_new_tokens=4096, use_cache=True)
    generated_ids_trimmed_instruct = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs_instruct.input_ids, generated_ids_instruct)]
    output_text_instruct = process1.batch_decode(
        generated_ids_trimmed_instruct, skip_special_tokens=False, clean_up_tokenization_spaces=False
    )[0]
end_time_4b = time.perf_counter() # 4B model çıkarım bitiş zamanı
elapsed_time_4b = end_time_4b - start_time_4b

print(f"\nModel 1 (Qwen Instruct) Ham Çıktısı:\n{output_text_instruct}\n")
qwen_instruct_tahminleri = parse_qwen_boxes(output_text_instruct, img_w, img_h)

# Model 3: QWEN3-VL-8B-Instruct TAHMİN ADIMI
messages_thinking = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": TEST_IMAGE_PATH},
            {"type": "text", "text": DETECTION_PROMPT_8B} # 8B modeli için yeni prompt
        ]
    }
]

text_thinking = process3.apply_chat_template(messages_thinking, tokenize=False, add_generation_prompt=True)
image_inputs_thinking, video_inputs_thinking = process_vision_info(messages_thinking)
inputs_thinking = process3(text=[text_thinking], images=image_inputs_thinking, videos=video_inputs_thinking, padding=True, return_tensors="pt")

if "pixel_values" in inputs_thinking:
    inputs_thinking["pixel_values"] = inputs_thinking["pixel_values"].to(torch.float32) # CPU için float32 kullan

inputs_thinking = inputs_thinking.to(device)

start_time_8b = time.perf_counter() # 8B model çıkarım başlangıç zamanı
with torch.no_grad():
    # torch.amp.autocast CPU'da desteklenmediği için kaldırıldı
    generated_ids_thinking = model3.generate(
            **inputs_thinking,
            max_new_tokens=4096,
            use_cache=True,
            do_sample=True, # Sampling'i etkinleştir
            temperature=1.2, # Daha çeşitli çıktılar için sıcaklığı artır
            top_p=0.9 # Yüksek olasılıklı token'lara odaklan
        )
    generated_ids_trimmed_thinking = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs_thinking.input_ids, generated_ids_thinking)]
    output_text_thinking = process3.batch_decode(
        generated_ids_trimmed_thinking, skip_special_tokens=False, clean_up_tokenization_spaces=False
    )[0]
end_time_8b = time.perf_counter() # 8B model çıkarım bitiş zamanı
elapsed_time_8b = end_time_8b - start_time_8b

print(f"\nModel 3 (Qwen 8B Instruct) Ham Çıktısı:\n{output_text_thinking}\n")
qwen_thinking_tahminleri = parse_qwen_boxes(output_text_thinking, img_w, img_h)

yolo_tahminleri = []

# CPU ortamında nvidia-smi çalışmadığı için kaldırıldı
# print("\n--- GPU Kullanımı (Inference Sonrası) ---")
# !nvidia-smi

# 3. GÖRSEL KUTULARI ÇİZDIRME (Qwen Instruct vs Qwen 8B Instruct)
plot_dual_boxes(TEST_IMAGE_PATH, qwen_instruct_tahminleri, qwen_thinking_tahminleri)

# 4. TEK OUTPUTTA METRİK RAPORLAMASI (Qwen Instruct vs Qwen 8B Instruct)
draw_and_analyze_dual(TEST_IMAGE_PATH, qwen_instruct_tahminleri, qwen_thinking_tahminleri, manuel_ground_truth)

print(f"\nModel 1 (Qwen3-VL 4B Instruct) Çıkarım Süresi: {elapsed_time_4b:.2f} saniye")
print(f"Model 3 (Qwen3-VL 8B Instruct) Çıkarım Süresi: {elapsed_time_8b:.2f} saniye")

ModuleNotFoundError: No module named 'qwen_vl_utils'